In [1]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import wbgapi as wb

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

def preprocess_data(df):
    """
    Preprocesses data for each country by handling NaN values and organizing data for analysis.
    
    This function processes the input DataFrame to prepare data for each country:
    1. Filters the data for each country
    2. Sorts the data by year
    3. Handles NaN values by either skipping countries with more than one NaN or interpolating single NaN values
    4. Creates a list of dictionaries containing the country name, original data, and cleaned values
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    
    Returns:
    list: A list of dictionaries, where each dictionary contains:
        - "country": The country name
        - "data": The original country data sorted by year
        - "values": Cleaned numeric values with NaNs handled
    """
    countries = df["country"].unique()
    preprocessed_data = []

    for country in countries:
        country_data = df[df["country"] == country].sort_values("year")
        values = country_data["value"].values.copy()

        # Handle NaN values
        nan_indices = np.isnan(values)
        if np.sum(nan_indices) > 1:
            continue

        if np.sum(nan_indices) == 1:
            nan_pos = np.where(nan_indices)[0][0]
            if nan_pos == 0:
                values[nan_pos] = values[nan_pos + 1]
            elif nan_pos == len(values) - 1:
                values[nan_pos] = values[nan_pos - 1]
            else:
                values[nan_pos] = (values[nan_pos - 1] + values[nan_pos + 1]) / 2

        preprocessed_data.append({
            "country": country,
            "data": country_data,
            "values": values
        })

    return preprocessed_data

def calculate_total_comparisons(df, min_length=3, max_length=5):
    """
    Calculate the total number of segment comparisons for progress tracking.
    
    This function computes the total number of comparisons that would be made 
    during the pattern extraction process. It considers all possible segment 
    lengths within the specified range and all country pairs.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    min_length (int): Minimum segment length to consider
    max_length (int): Maximum segment length to consider
    
    Returns:
    int: Total number of comparisons that would be performed
    """
    preprocessed_data = preprocess_data(df)
    total_comparisons = 0

    for i, country_a in enumerate(preprocessed_data):
        values_a = country_a["values"]

        for segment_length in range(min_length, max_length + 1):
            if len(values_a) < segment_length:
                continue

            num_segments_a = len(values_a) - segment_length + 1
            num_segments_b = sum(
                len(country_b["values"]) - segment_length + 1
                for country_b in preprocessed_data[i+1:]
                if len(country_b["values"]) >= segment_length
            )
            total_comparisons += num_segments_a * num_segments_b

    return total_comparisons

class Indicator:
    def __init__(self):
        self.id                   = ""
        self.data                 = pd.DataFrame()
        self.indicator_name       = ""
        self.concept              = ""
        self.license_type         = ""
        self.license_url          = ""
        self.long_definition      = ""
        self.othernotes           = ""
        self.periodicity          = ""
        self.related_source_links = ""
        self.source               = ""
        self.topic                = ""

def get_indicator(indicator_id):
    """
    Example output:
    id:                   SP.POP.TOTL
    data columns:         ['YR1960', 'YR1961', 'YR1962', 'YR1963', 'YR1964', 'YR1965', 'YR1966',
                           'YR1967', 'YR1968', 'YR1969', 'YR1970', 'YR1971', 'YR1972', 'YR1973',
                           'YR1974', 'YR1975', 'YR1976', 'YR1977', 'YR1978', 'YR1979', 'YR1980',
                           'YR1981', 'YR1982', 'YR1983', 'YR1984', 'YR1985', 'YR1986', 'YR1987',
                           'YR1988', 'YR1989', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994',
                           'YR1995', 'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001',
                           'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008',
                           'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015',
                           'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022',
                           'YR2023']
    data rows:            266
    name:                 Population, total
    concept:              Series
    license_type:         CC BY-4.0
    license_url:          https://datacatalog.worldbank.org/public-licenses#cc-by
    long_definition:      Total population is based on the de facto definition of population, 
                          which counts all residents regardless of legal status or citizenship. 
                          The values shown are midyear estimates.
    othernotes:           
    periodicity:          Annual
    related_source_links: 
    source:               (1) United Nations Population Division. World Population Prospects:
                          2024 Revision; (2) Statistical databases and publications from national
                          statistical offices; (3) Eurostat: Demographic Statistics;
                          (4) United Nations Statistics Division. Population and Vital Statistics
                          Report (various years).
    topic:                Health: Population: Structure
    """
    indicator          = Indicator()
    indicator_metadata = wb.series.metadata.get(indicator_id)
    indicator.id       = indicator_id
    indicator.data     = wb.data.DataFrame(indicator_id)

    if "IndicatorName" in list(indicator_metadata.metadata.keys()):
        indicator.name = indicator_metadata.metadata["IndicatorName"]
    else:
        indicator.name = ""

    indicator.concept = indicator_metadata.concept

    if "License_Type" in list(indicator_metadata.metadata.keys()):
        indicator.license_type = indicator_metadata.metadata["License_Type"]
    else:
        indicator.license_type = ""

    if "License_URL" in list(indicator_metadata.metadata.keys()):
        indicator.license_url = indicator_metadata.metadata["License_URL"]
    else:
        indicator.license_url = ""

    if "Longdefinition" in list(indicator_metadata.metadata.keys()):
        indicator.long_definition = indicator_metadata.metadata["Longdefinition"]
    else:
        indicator.long_definition = ""

    if "Othernotes" in list(indicator_metadata.metadata.keys()):
        indicator.othernotes = indicator_metadata.metadata["Othernotes"]
    else:
        indicator.othernotes = ""

    if "Periodicity" in list(indicator_metadata.metadata.keys()):
        indicator.periodicity = indicator_metadata.metadata["Periodicity"]
    else:
        indicator.periodicity = ""

    if "Relatedsourcelinks" in list(indicator_metadata.metadata.keys()):
        indicator.related_source_links = indicator_metadata.metadata["Relatedsourcelinks"]
    else:
        indicator.related_source_links = ""

    if "Source" in list(indicator_metadata.metadata.keys()):
        indicator.source = indicator_metadata.metadata["Source"]
    else:
        indicator.source = ""

    if "Topic" in list(indicator_metadata.metadata.keys()):
        indicator.topic = indicator_metadata.metadata["Topic"]
    else:
        indicator.topic = ""

    return indicator

#### Show Parquet as DF

In [6]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\pattern_maker_output\part.2.parquet")
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,average_correlation,scdi_count
0,74,79,3,1960,2021,0.999951,1
1,74,65,3,1960,1968,0.537074,1
2,74,85,3,1960,2015,0.763293,1
3,74,103,3,1960,2001,0.999999,1
4,74,118,3,1960,2018,0.998855,2
...,...,...,...,...,...,...,...
3487236,155,158,58,1966,1960,0.945776,1
3487237,155,232,58,1966,1960,0.963414,1
3487238,155,222,59,1965,1965,0.963763,1
3487239,155,199,62,1962,1961,0.957879,1


#### Shrinking

In [18]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL\SP.POP.TOTL.parquet")
output_path = r"/workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet"

# Select 5 countries to keep
target_countries = ["United States", "United Kingdom", "Germany", "France", "Canada", "Australia", "Japan", "India", "China", "Brazil", "Mexico", "South Africa", "Nigeria", "Egypt", "Turkiye", "Russia", "Italy", "Spain", "Argentina", "South Korea", "Indonesia", "Pakistan", "Bangladesh", "Thailand", "Philippines", "Vietnam", "Kenya", "Tanzania", "Uganda", "Ghana", "Senegal", "Morocco", "Algeria", "Israel", "Saudi Arabia", "Iran", "Iraq", "Afghanistan", "Poland", "Hungary", "Czech", "Greece", "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", "Finland", "Switzerland", "Austria"]

# Filter the DataFrame
filtered_df = df[df["country"].isin(target_countries)]

filtered_df.to_parquet(output_path, index = False, compression = "snappy")
print(f"Shrinked Parquet file saved to {output_path}")
filtered_df

Shrinked Parquet file saved to /workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet


,country,year,value
128,Afghanistan,1960,9035043.0
129,Afghanistan,1961,9214083.0
130,Afghanistan,1962,9404406.0
131,Afghanistan,1963,9604487.0
132,Afghanistan,1964,9814318.0
...,...,...,...
16891,South Africa,2019,59587885.0
16892,South Africa,2020,60562381.0
16893,South Africa,2021,61502603.0
16894,South Africa,2022,62378410.0


In [5]:
from typing import Sequence


def process_sequence(seq: Sequence[int]) -> None:
    for item in seq:
        print(item)

# Populate a list and pass it to the function
my_sequence = [1, 2, 3]
process_sequence(my_sequence)

1
2
3


#### Calculating Comaprisons

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\NY.GDP.MKTP.KD.ZG\NY.GDP.MKTP.KD.ZG.parquet")

# Example usage:
total = calculate_total_comparisons(df)
print(f"Total comparisons: {total}")

Total comparisons: 60960900


#### Size Saving Comparison

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked_sdcis.parquet")
print(df.head())

   country_a_id  country_b_id  length  start_year_a  start_year_b  correlation
0            45            81       3          1960          1977     0.945475
1            45            81       3          1961          1960     0.970550
2            45            81       3          1961          1961     0.964687
3            45            81       3          1961          1962     0.948679
4            45            81       3          1961          1963     0.965124


In [15]:
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,45,81,3,1960,1977,0.945475
1,45,81,3,1961,1960,0.970550
2,45,81,3,1961,1961,0.964687
3,45,81,3,1961,1962,0.948679
4,45,81,3,1961,1963,0.965124
...,...,...,...,...,...,...
82764,110,236,5,2005,2015,0.958875
82765,110,236,5,2005,2017,0.943587
82766,110,236,5,2005,2018,0.978710
82767,110,236,5,2005,2019,0.901636


In [13]:
df

,country_a,country_b,length,start_year_a,start_year_b,correlation,data_segment_a,data_segment_b
0,China,Germany,3,1960,1977,0.945475,"[667070000.0, 660330000.0, 665770000.0]","[78159814.0, 78091820.0, 78126350.0]"
1,China,Germany,3,1961,1960,0.970550,"[660330000.0, 665770000.0, 682335000.0]","[72814900.0, 73377632.0, 74025784.0]"
2,China,Germany,3,1961,1961,0.964687,"[660330000.0, 665770000.0, 682335000.0]","[73377632.0, 74025784.0, 74714353.0]"
3,China,Germany,3,1961,1962,0.948679,"[660330000.0, 665770000.0, 682335000.0]","[74025784.0, 74714353.0, 75318337.0]"
4,China,Germany,3,1961,1963,0.965124,"[660330000.0, 665770000.0, 682335000.0]","[74714353.0, 75318337.0, 75963695.0]"
...,...,...,...,...,...,...,...,...
82764,Japan,United States,5,2005,2015,0.958875,"[127773000.0, 127854000.0, 128001000.0, 128063...","[320738994.0, 323071755.0, 325122128.0, 326838..."
82765,Japan,United States,5,2005,2017,0.943587,"[127773000.0, 127854000.0, 128001000.0, 128063...","[325122128.0, 326838199.0, 328329953.0, 331526..."
82766,Japan,United States,5,2005,2018,0.978710,"[127773000.0, 127854000.0, 128001000.0, 128063...","[326838199.0, 328329953.0, 331526933.0, 332048..."
82767,Japan,United States,5,2005,2019,0.901636,"[127773000.0, 127854000.0, 128001000.0, 128063...","[328329953.0, 331526933.0, 332048977.0, 333271..."


#### Testing WB Extractor

- PV.EST:             25,016,070 - Political Stability
- NY.GDP.MKTP.CD:     68,162,325 - GDP (current US$)
- SP.DYN.LE00.IN:    217,120,200 - Life expectancy at birth, total (years)
- SE.PRM.ENRR:           936,600 - School enrollment, primary (% gross)
- SL.EMP.TOTL.SP.ZS:  45,988,040 - Employment to population ratio, 15+, total (%) (modeled ILO estimate)
- SL.UEM.TOTL.ZS:     45,988,040 - Unemployment, total (% of total labor force) (modeled ILO estimate)
- NY.GDP.PCAP.CD:     68,162,325 - GDP per capita (current US$)
- MS.MIL.MPRT.KD:      7,033,950 - Arms imports (SIPRI trend indicator values)

In [129]:
class Indicator:
    def __init__(self):
        self.id                   = ""
        self.data                 = pd.DataFrame()
        self.indicator_name       = ""
        self.concept              = ""
        self.license_type         = ""
        self.license_url          = ""
        self.long_definition      = ""
        self.othernotes           = ""
        self.periodicity          = ""
        self.related_source_links = ""
        self.source               = ""
        self.topic                = ""

def get_indicator(indicator_id):
    """
    Example output:
    id:                   SP.POP.TOTL
    data columns:         ['YR1960', 'YR1961', 'YR1962', 'YR1963', 'YR1964', 'YR1965', 'YR1966',
                           'YR1967', 'YR1968', 'YR1969', 'YR1970', 'YR1971', 'YR1972', 'YR1973',
                           'YR1974', 'YR1975', 'YR1976', 'YR1977', 'YR1978', 'YR1979', 'YR1980',
                           'YR1981', 'YR1982', 'YR1983', 'YR1984', 'YR1985', 'YR1986', 'YR1987',
                           'YR1988', 'YR1989', 'YR1990', 'YR1991', 'YR1992', 'YR1993', 'YR1994',
                           'YR1995', 'YR1996', 'YR1997', 'YR1998', 'YR1999', 'YR2000', 'YR2001',
                           'YR2002', 'YR2003', 'YR2004', 'YR2005', 'YR2006', 'YR2007', 'YR2008',
                           'YR2009', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015',
                           'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022',
                           'YR2023']
    data rows:            266
    name:                 Population, total
    concept:              Series
    license_type:         CC BY-4.0
    license_url:          https://datacatalog.worldbank.org/public-licenses#cc-by
    long_definition:      Total population is based on the de facto definition of population, 
                          which counts all residents regardless of legal status or citizenship. 
                          The values shown are midyear estimates.
    othernotes:           
    periodicity:          Annual
    related_source_links: 
    source:               (1) United Nations Population Division. World Population Prospects:
                          2024 Revision; (2) Statistical databases and publications from national
                          statistical offices; (3) Eurostat: Demographic Statistics;
                          (4) United Nations Statistics Division. Population and Vital Statistics
                          Report (various years).
    topic:                Health: Population: Structure
    """
    indicator          = Indicator()
    try:
        indicator_metadata = wb.series.metadata.get(indicator_id)
    except Exception as e:
        print("Error caught when extracting indicator metadata: ", str(e))
    indicator.id       = indicator_id
    indicator.data     = wb.data.DataFrame(indicator_id)

    if "IndicatorName" in list(indicator_metadata.metadata.keys()):
        indicator.name = indicator_metadata.metadata["IndicatorName"]
    else:
        indicator.name = ""

    indicator.concept = indicator_metadata.concept

    if "License_Type" in list(indicator_metadata.metadata.keys()):
        indicator.license_type = indicator_metadata.metadata["License_Type"]
    else:
        indicator.license_type = ""

    if "License_URL" in list(indicator_metadata.metadata.keys()):
        indicator.license_url = indicator_metadata.metadata["License_URL"]
    else:
        indicator.license_url = ""

    if "Longdefinition" in list(indicator_metadata.metadata.keys()):
        indicator.long_definition = indicator_metadata.metadata["Longdefinition"]
    else:
        indicator.long_definition = ""

    if "Othernotes" in list(indicator_metadata.metadata.keys()):
        indicator.othernotes = indicator_metadata.metadata["Othernotes"]
    else:
        indicator.othernotes = ""

    if "Periodicity" in list(indicator_metadata.metadata.keys()):
        indicator.periodicity = indicator_metadata.metadata["Periodicity"]
    else:
        indicator.periodicity = ""

    if "Relatedsourcelinks" in list(indicator_metadata.metadata.keys()):
        indicator.related_source_links = indicator_metadata.metadata["Relatedsourcelinks"]
    else:
        indicator.related_source_links = ""

    if "Source" in list(indicator_metadata.metadata.keys()):
        indicator.source = indicator_metadata.metadata["Source"]
    else:
        indicator.source = ""

    if "Topic" in list(indicator_metadata.metadata.keys()):
        indicator.topic = indicator_metadata.metadata["Topic"]
    else:
        indicator.topic = ""

    return indicator

###########################
indicator = get_indicator("PV.EST")

indicator.data.head()

def remove_all_nan_year_columns(df):
    """TODO"""

    for year_col in list(df):
        nan_status = list(set(np.isnan(list(df[year_col]))))
        if len(nan_status) == 1 and str(nan_status[0]) == "True":
            df = df.drop([year_col], axis = 1)

    return df

def pivot_dataframe(df):
    """
    Reads a wide-format CSV file and pivots it into a long format.

    Parameters:
        file_path (str): Path to the input CSV file.

    Returns:
        pd.DataFrame: A DataFrame in long format with columns:
                      - 'economy': The economy code (e.g., ABW, AFE, AFG).
                      - 'year': The year extracted from column names (e.g., 1960, 1961).
                      - 'value': The corresponding value for the economy and year.
    """
    # Melt the DataFrame to convert year columns into rows
    df["economy"] = df.index
    df_long = pd.melt(
        df,
        id_vars=['economy'],  # Keep 'economy' as an identifier
        var_name='year',      # New column for years
        value_name='value'    # New column for values
    )

    # Extract the numeric part of the year (remove 'YR' prefix)
    df_long['year'] = df_long['year'].str.extract(r'(\d+)').astype(int)

    # Sort the DataFrame by economy and year for better readability
    df_long = df_long.sort_values(by=['economy', 'year']).reset_index(drop=True)

    return df_long

def rename_columns(df, column_mapping = None):
    if column_mapping is None:
        column_mapping = {
            'economy': 'country',
        }
    return df.rename(columns=column_mapping)

## TODO: figure out a better way to link "countries.csv" with this function,
## rather than reading it multiple times and taking out the "Unnamed" columns.
## TODO centralize paths
def clean_country_names(input_df):
    """
    Cleans the 'country' column of the input DataFrame by replacing codes with their corresponding country names.
    
    Parameters:
        input_df (pd.DataFrame): The input DataFrame with a 'country' column to be cleaned.
    
    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: A DataFrame with the 'country' column replaced by country names.
            - list: A list of unmatched values from the 'country' column.
    """
    # Step 1: Load the reference CSV file
    reference_df = pd.read_csv(r"/workspace/qgi\qgi-4.0\data\countries.csv") # r"/workspaces/qgi-4.0/data/countries.csv"
    reference_df = reference_df.loc[:, ~reference_df.columns.str.contains("Unnamed")]

    # Ensure 'name_1' exists in the reference DataFrame
    if 'name_1' not in reference_df.columns:
        raise KeyError("The column 'name_1' is missing in the reference data.")

    # Step 2: Create a mapping dictionary
    mapping = {}
    for col in reference_df.columns:
        if col != 'name_1':  # Skip the 'name_1' column itself
            temp_mapping = reference_df.dropna(subset=[col]).set_index(col)['name_1'].to_dict()
            mapping.update(temp_mapping)

    # Step 3: Replace values in the 'country' column using the mapping
    unmatched_values = []  # List to store unmatched values
    input_df['country'] = input_df['country'].apply(
        lambda x: mapping.get(x, x)  # Replace if found in mapping, otherwise keep original
    )
    # Identify unmatched values
    unmatched_values = [
        value for value in input_df['country'] if value not in mapping.values() and value not in mapping.keys()
    ]

    return input_df, list(set(unmatched_values))

def delete_rows_by_country(input_df, countries_to_delete):
    """
    Deletes rows from the input DataFrame where the 'country' column matches any value in the given list.

    Parameters:
        input_df (pd.DataFrame): The input DataFrame.
        countries_to_delete (list): A list of country names or codes to delete.

    Returns:
        pd.DataFrame: A DataFrame with the specified rows removed.
    """
    # Filter out rows where the 'country' column matches any value in the list
    filtered_df = input_df[~input_df['country'].isin(countries_to_delete)]
    
    return filtered_df

indicator.data = remove_all_nan_year_columns(indicator.data)
indicator.data = pivot_dataframe(indicator.data)
indicator.data = rename_columns(indicator.data)
indicator.data, unmatched_countries = clean_country_names(indicator.data)
indicator.data = delete_rows_by_country(indicator.data, unmatched_countries)

print(calculate_total_comparisons(indicator.data))

indicator.data

25016070


,country,year,value
0,Aruba,1996,NaN
1,Aruba,1998,NaN
2,Aruba,2000,NaN
3,Aruba,2002,NaN
4,Aruba,2003,NaN
...,...,...,...
6645,Zimbabwe,2019,-0.943303
6646,Zimbabwe,2020,-1.052743
6647,Zimbabwe,2021,-0.954443
6648,Zimbabwe,2022,-0.894974


In [ ]:
indicator = get_indicator("SI.POV.DDAY")

indicator.data.head()

,YR1960,YR1961,YR1962,YR1963,YR1964,YR1965,YR1966,YR1967,YR1968,YR1969,...,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
economy,,,,,,,,,,,,,,,,,,,,,
ABW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,31.1,NaN,NaN,NaN,NaN,NaN
